This notebook retrains PhishGuard's phishing detector after discovering the original model achieved ~100% test accuracy but failed catastrophically in production — flagging legitimate sites like github.com as phishing. Investigation revealed the training data's legitimate URLs were all bare homepages, while phishing URLs had paths — the model learned "has path = phishing," not real signal. This notebook retrains on a dataset where legitimate URLs have realistic paths, using the same feature extractor as the production API to prevent the mismatch from recurring.

In [1]:
# Load the dataset
import pandas as pd

df = pd.read_csv("../data/malicious_phish.csv")
print(df.shape)
print(df["type"].value_counts())

(651191, 2)
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


In [2]:
# Binary task: benign vs phishing (defacement and malware are different detection problems)
# We downsample benign to match phishing so the model cannot win by majority-guessing. So it becomes balanced and evens out, or else we have class imbalance.
df = df[df["type"].isin(["benign", "phishing"])].copy()
df["label"] = (df["type"] == "phishing").astype(int)  # 1 = phishing

n_phish = (df["label"] == 1).sum()
benign_sample = df[df["label"] == 0].sample(n=n_phish, random_state=42)
df_balanced = pd.concat([df[df["label"] == 1], benign_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced["label"].value_counts())

label
0    94111
1    94111
Name: count, dtype: int64


In [3]:
# Remove http:// and https:// from URLs before extracting features — this was a dataset quirk, not something that actually signals phishing.
import re

def strip_scheme(url):
    return re.sub(r"^https?://", "", url, flags=re.IGNORECASE)

df_balanced["url_clean"] = df_balanced["url"].apply(strip_scheme)
print(df_balanced[["url", "url_clean"]].head())

                                                 url  \
0  legaltimes.typepad.com/blt/2011/05/dc-superior...   
1  local.yahoo.com/info-20135686-flagstaff-lawyer...   
2                               amcclain40.myjino.ru   
3  http://doodle.com/pt/premium?utm_source=doodle...   
4  http://torcache.net/torrent/210556CAB0730F31EF...   

                                           url_clean  
0  legaltimes.typepad.com/blt/2011/05/dc-superior...  
1  local.yahoo.com/info-20135686-flagstaff-lawyer...  
2                               amcclain40.myjino.ru  
3  doodle.com/pt/premium?utm_source=doodle&utm_me...  
4  torcache.net/torrent/210556CAB0730F31EF4E2C147...  


In [4]:
# Extract features using the bakcned's extractor.py so training and the live API use identical logic - eliminating train/serve/skew
import sys
sys.path.append("../backend")

from app.features.extractor import extract_features, FEATURE_ORDER

print(FEATURE_ORDER)

['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'LetterRatioInURL', 'DegitRatioInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL']


In [5]:
# Check for URLs that break urlparse before running extraction on the full dataset — some rows turn out to be corrupted binary garbage, not real URLs, and crash the parser with an IPv6 error.
from urllib.parse import urlparse

bad_rows = []
for i, url in enumerate(df_balanced["url_clean"]):
    test_url = url if "://" in url else "http://" + url
    try:
        urlparse(test_url)
    except ValueError as e:
        bad_rows.append((i, url, str(e)))

print(len(bad_rows))
for row in bad_rows[:10]:
    print(row)

17
(32666, '=\x9dRã\x0fmôj³{\x94è\x95!ÀM\x97¶6<\x9cN>\x9ew\x85¼Cf\x11£]\x1b4ÍnÝÌ\x9c', 'Invalid IPv6 URL')
(41305, '¨R\x98ÊÃ\x86ûaCóÞit×ßÂe-DÖ\x8bØ+9YèÌçÏ\x97¯·\x04"0£ÙÕ.0ößF«7¹N\x89R\x1c\x04Ù{ccÉÄãéçx[Ä6a\x1a5Ñ³LÖíÜÉÀ£\x9dÒma¥yRX\x03\x9a*0ÅÝ7×Ê\x83ÁÌ\x05\x05o«Õs¶\x8d0k\x90dèÑ&\x83\x1cÄ\x10"Ï¨mZ\'àD\x8fM×ñ\x01XÚÒK"päî±h¬\x83cAÊeK@4r"^\'ÓFþ1*Ë\x8e\x1d\x8dË PÞô;õ\x9c$úàÑ@\x87þ=êWÑ"Ãhñ\x05\x18®ç^\x18\x11«Ýó^ç\x1a\x1fRú\x8eUJ\x14.<6C\x19\x94y\x1a\x9fÜ\x94FØrÿV2ôæý\x89\x03Zãii\x16\x93I\x0e\x8ab;\x13\x16¨Ë\x97\x9cµu^Í\x99V\x90y)\x9d\xadè»âýº\x01+\x9f\x94S\x99Ö\x1e\x17á\x10\x95\x03Ãì?\x1få6åÔ/', 'Invalid IPv6 URL')
(41536, '1]Î¼0#W»æ½Î4>¥õ\x1cª\x94(\\xl\x863(ò5?¹(\x8d°åþ¬eéÍû\x12\x06µÆÒÒ-&\x92\x1d\x14Äv&-Q\x97/9jê½\x9b2\xad òS;[ÑwÅût\x02W?(§3¬</Â!*\x07\x87Ø~?ÊmË¨^XV\x9c¹µÂ\x92¦\x183¨|[÷4\x10fÈë<\t\x81ô·»n³H\x96\x99éÜúÂÒá/Wîà.K3q4:å\x81)¿®I\x13K.°x±\x8e&\x0fR6\x87\x90¹àÄ\x01#\x1f|9³¢Ü\x89\x87\x94ù\x94ñ\x14\x19~3', 'Invalid IPv6 URL')
(58412, '\x90Æe\x1eF§÷%\x11¶\x1c¿Õ\x8c½9¿b@Ö¸ÚZE¤ÒC¢\x98\x8e

In [6]:
# Drop the 17 corrupted rows and reset the index.
bad_indices = [row[0] for row in bad_rows]
df_balanced = df_balanced.drop(index=bad_indices).reset_index(drop=True)
print(df_balanced.shape)

(188205, 4)


In [7]:
# Extract all 16 features for every URL using the shared extractor.
features_list = df_balanced["url_clean"].apply(extract_features)
X = pd.DataFrame(features_list.tolist(), columns=FEATURE_ORDER)
X["label"] = df_balanced["label"].values

print(X.shape)
print(X.head())

(188205, 17)
   URLLength  DomainLength  IsDomainIP  CharContinuationRate  \
0        113            22           0              0.115044   
1         78            15           0              0.115385   
2         27            20           0              0.370370   
3         87            10           0              0.126437   
4        161            12           0              0.248447   

   TLDLegitimateProb  URLCharProb  TLDLength  NoOfSubDomain  \
0           0.510085     0.040489          3              2   
1           0.510085     0.032832          3              2   
2           0.006615     0.041954          2              2   
3           0.510085     0.038877          3              1   
4           0.029648     0.034720          3              1   

   NoOfObfuscatedChar  ObfuscationRatio  LetterRatioInURL  DegitRatioInURL  \
0                   0               0.0          0.787611         0.053097   
1                   0               0.0          0.730769         0

In [8]:
# Split into train/test sets, then train a Random Forest — same model type as v1, so we can compare fairly.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_features = X[FEATURE_ORDER]
y = X["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.89      0.89     18822
           1       0.89      0.89      0.89     18819

    accuracy                           0.89     37641
   macro avg       0.89      0.89      0.89     37641
weighted avg       0.89      0.89      0.89     37641



In [9]:
# Check PhiUSIIL's column names and label encoding before filtering to phishing URLs
phiusiil = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
print(phiusiil.columns.tolist())
print(phiusiil.head())

['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']
     FILENAME                                 URL  URLLength  \
0  521848.txt    

In [10]:
# Check PhiUSIIL's column names and label encoding before filtering to phishing URLs.
import os
print(os.listdir("../data"))

['malicious_phish.csv', 'phishing_url_dataset_raw.csv', 'PhiUSIIL_Phishing_URL_Dataset.csv']


In [11]:
# Load PhiUSIIL and inspect its columns - this will be the out-of-distribution test set.
phiusiil = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
print(phiusiil.columns.tolist())
print(phiusiil.head())

['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']
     FILENAME                                 URL  URLLength  \
0  521848.txt    

In [12]:
# Confirm label encoding, then filter to phishing URLs only (label == 0 in PhiUSIIL).
print(phiusiil["label"].value_counts())

phiusiil_phish = phiusiil[phiusiil["label"] == 0]
print(phiusiil_phish.shape)
print(phiusiil_phish["URL"].head())

label
1    134850
0    100945
Name: count, dtype: int64
(100945, 56)
11                   http://www.teramill.com
20               http://www.f0519141.xsph.ru
21                  http://www.shprakserf.gq
27    https://service-mitld.firebaseapp.com/
28         http://www.kuradox92.lima-city.de
Name: URL, dtype: str


In [13]:
# Extract features from PhiUSIIL phishing URLs and run them through the model — measures generalization to phishing URLs it's never seen before.
phiusiil_urls_clean = phiusiil_phish["URL"].apply(strip_scheme)
phiusiil_features = phiusiil_urls_clean.apply(extract_features)
X_phiusiil = pd.DataFrame(phiusiil_features.tolist(), columns=FEATURE_ORDER)

preds = model.predict(X_phiusiil)
print(f"Caught as phishing: {preds.sum()} / {len(preds)} ({preds.mean()*100:.1f}%)")

Caught as phishing: 87072 / 100945 (86.3%)


86.3% of PhiUSIIL's phishing URLs (a dataset the model never trained on) are correctly caught. Compare this to v1's ~100% test accuracy that failed on real traffic — this model's lower, honest number reflects real generalization instead of a memorized shortcut.

In [14]:
# Sanity check: verify the model doesn't flag legitimate URLs with paths as phishing — the exact failure mode of v1.
test_urls = [
    "github.com/AhmedNaoum97",
    "google.com/search?q=phishing+detection",
    "en.wikipedia.org/wiki/Phishing",
    "stackoverflow.com/questions/12345",
    "linkedin.com/in/ahmednaoum97",
]

test_clean = [strip_scheme(u) for u in test_urls]
test_features = pd.DataFrame([extract_features(u) for u in test_clean], columns=FEATURE_ORDER)
test_preds = model.predict(test_features)

for url, pred in zip(test_urls, test_preds):
    print(f"{'PHISHING' if pred == 1 else 'legit':10} {url}")

legit      github.com/AhmedNaoum97
legit      google.com/search?q=phishing+detection
PHISHING   en.wikipedia.org/wiki/Phishing
legit      stackoverflow.com/questions/12345
legit      linkedin.com/in/ahmednaoum97


4 of 5 legitimate URLs correctly classified — including github.com/AhmedNaoum97, the exact case that broke v1. The one false positive (Wikipedia's own article on phishing) is a reasonable edge case: the URL's path literally contains the word 'phishing' and is unusually long, both signals the model has learned to weight. This is expected imperfection from a single-signal, URL-only detector — not a repeat of the original bug.

In [18]:
# Retrain with flipped labels so 0 = phishing, matching predictor.py's convention.
y_flipped = 1 - y  # 0 = phishing, 1 = legit — matches predictor.py's convention

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_flipped, test_size=0.2, random_state=42, stratify=y_flipped
)

model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.89      0.90     18819
           1       0.89      0.90      0.90     18822

    accuracy                           0.90     37641
   macro avg       0.90      0.90      0.90     37641
weighted avg       0.90      0.90      0.90     37641



In [19]:
# Serialize the retrained model and feature column order for deployment. Save the trained model to a file, so the backend can load and use it later basically.
import joblib

joblib.dump(model, "../backend/app/ml/phishguard_model_url_only.pkl")
print("Saved to correct filename with corrected label convention.")

Saved to correct filename with corrected label convention.


In [20]:
# Sanity check the flipped model directly, before trusting the API
for url, pred in zip(test_urls, model.predict(test_features)):
    print(f"{'flip-class 0 (phishing)' if pred == 0 else 'flip-class 1 (legit)':30} {url}")

flip-class 1 (legit)           github.com/AhmedNaoum97
flip-class 1 (legit)           google.com/search?q=phishing+detection
flip-class 1 (legit)           en.wikipedia.org/wiki/Phishing
flip-class 1 (legit)           stackoverflow.com/questions/12345
flip-class 1 (legit)           linkedin.com/in/ahmednaoum97
